In [9]:
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from som_model import GeneralCLASSOM

LOAD CSV FILE WITH TRAIN DATA
check shape and featuresname

In [10]:
df = pd.read_csv("../data/processed/eda_ppg_features/features_all.csv")
df.describe()




,HR,HRV_RMSSD,RR_diff_std,HRV_SDNN,RR_mean,RR_std,RR_min,RR_max,RR_count,RR_quality,RR_valid,time_sec,SC_PH,SC_RR,EDA_slope
count,5780.000000,5780.000000,5780.000000,5780.000000,5780.000000,5780.000000,5780.000000,5780.000000,5780.000000,5780.000000,5780.0,5780.000000,5780.000000,5780.000000,5780.000000
mean,80.025798,47.553638,47.397282,62.416303,768.121482,62.006252,613.227509,937.557958,79.113668,0.946798,1.0,1108.575260,0.597967,0.110865,0.000370
std,11.424437,16.090708,15.947467,24.193318,102.474180,24.020108,106.153195,127.883394,10.841656,0.082160,0.0,647.975482,2.162083,0.086935,0.012616
min,54.150973,8.767506,8.722435,15.359078,449.492188,15.287805,405.000000,510.000000,48.000000,0.450704,1.0,0.000000,0.000000,0.000000,-0.780189
25%,72.290269,35.036242,35.012774,45.011288,699.926813,44.713765,540.000000,855.000000,72.000000,0.926471,1.0,555.000000,0.031479,0.050000,-0.000571
50%,78.520949,47.392854,47.326500,58.716643,766.730769,58.308783,615.000000,945.000000,78.000000,0.986111,1.0,1110.000000,0.120183,0.100000,0.000272
75%,86.429098,59.980737,59.738815,76.520495,830.684932,76.035403,690.000000,1025.000000,85.000000,1.000000,1.0,1665.000000,0.415318,0.166667,0.001445
max,141.942360,94.688192,94.607741,183.023187,1093.750000,181.465514,990.000000,1195.000000,128.000000,1.000000,1.0,2265.000000,61.832978,0.983333,0.073306


In [11]:
df["SC_PH"] = np.log1p(df["SC_PH"])
low, high = df["SC_PH"].quantile([0.01, 0.99])
df["SC_PH"] = df["SC_PH"].clip(low, high)

# --- SC_RR ---
low, high = df["SC_RR"].quantile([0.01, 0.99])
df["SC_RR"] = df["SC_RR"].clip(low, high)

# --- SLOPE ---
low, high = df["EDA_slope"].quantile([0.01, 0.99])
df["EDA_slope"] = df["EDA_slope"].clip(low, high)

df = df[df["RR_quality"] > 0.7]

df.describe()

,HR,HRV_RMSSD,RR_diff_std,HRV_SDNN,RR_mean,RR_std,RR_min,RR_max,RR_count,RR_quality,RR_valid,time_sec,SC_PH,SC_RR,EDA_slope
count,5651.000000,5651.000000,5651.000000,5651.000000,5651.000000,5651.000000,5651.000000,5651.00000,5651.000000,5651.000000,5651.0,5651.000000,5651.000000,5651.000000,5651.000000
mean,79.903570,47.019125,46.882389,60.890892,768.435271,60.491034,616.922669,933.92143,79.172536,0.953987,1.0,1101.994337,0.290951,0.109140,0.000633
std,11.357451,15.820953,15.699463,22.080268,103.288358,21.920536,104.237293,126.59716,10.926616,0.067164,0.0,644.845577,0.424058,0.079746,0.004599
min,54.150973,8.767506,8.722435,15.359078,449.492188,15.287805,405.000000,510.00000,48.000000,0.701299,1.0,0.000000,0.000258,0.000000,-0.017779
25%,72.150531,34.733582,34.680067,44.675961,699.186047,44.395754,545.000000,855.00000,72.000000,0.934211,1.0,540.000000,0.031997,0.050000,-0.000576
50%,78.445344,46.788531,46.722134,58.068509,766.666667,57.668504,620.000000,940.00000,78.000000,0.986486,1.0,1095.000000,0.116933,0.100000,0.000268
75%,86.304307,59.149089,58.947479,75.026828,831.736111,74.507921,695.000000,1020.00000,85.000000,1.000000,1.0,1650.000000,0.355808,0.166667,0.001455
max,134.242511,93.271439,92.717763,135.851397,1093.750000,134.486022,990.000000,1195.00000,128.000000,1.000000,1.0,2265.000000,2.167563,0.366667,0.020909


In [12]:

df_som = df[['HR', 'HRV_RMSSD','SC_PH', 'SC_RR', 'EDA_slope']].copy()
# df_som = df.copy()
print("Shape:", df_som.shape)
print("\nColumns:")
print(df_som.columns)

print("\nDescribe:")
print(df_som.describe())

Shape: (5651, 5)

Columns:
Index(['HR', 'HRV_RMSSD', 'SC_PH', 'SC_RR', 'EDA_slope'], dtype='str')

Describe:
                HR    HRV_RMSSD        SC_PH        SC_RR    EDA_slope
count  5651.000000  5651.000000  5651.000000  5651.000000  5651.000000
mean     79.903570    47.019125     0.290951     0.109140     0.000633
std      11.357451    15.820953     0.424058     0.079746     0.004599
min      54.150973     8.767506     0.000258     0.000000    -0.017779
25%      72.150531    34.733582     0.031997     0.050000    -0.000576
50%      78.445344    46.788531     0.116933     0.100000     0.000268
75%      86.304307    59.149089     0.355808     0.166667     0.001455
max     134.242511    93.271439     2.167563     0.366667     0.020909


FUNCTION FOR TOPOGRAPIC_ERROR 

CREATE SOM MODEL PARAMETERS

OUTPUT: SUMMARY OF DIFFERENT PARAMETERS

In [13]:
som_clas_200 = GeneralCLASSOM(

    x=22,
    y=22,
    input_len=5,

    sigma=5,
    learning_rate=0.1,

    random_seed=42
)

som_clas_200.get_summary()

{'x': 22,
 'y': 22,
 'input_len': 5,
 'sigma': 5,
 'learning_rate': 0.1,
 'decay_function': 'asymptotic_decay',
 'neighborhood_function': 'gaussian',
 'topology': 'rectangular',
 'activation_distance': 'euclidean',
 'sigma_decay_function': 'asymptotic_decay',
 'random_seed': 42}

In [15]:
som_clas_200.inspect_bmu_distribution(df_som)

🔍 Total samples: 5651
🔍 Unique BMUs: 148
🔍 Map size: 484


array([[13,  9],
       [14, 10],
       [14, 10],
       ...,
       [ 8, 13],
       [ 9, 12],
       [ 9, 12]], shape=(5651, 2))

TRAIN MODELL 

Change parameters

iterations: Batch train, 1 datapoint on each iteration. 

pca

In [16]:
som_clas_200.train(

   df_som,

    iterations=60000,
    mode="batch",

    use_pca=True,
    verbose=True
)

 [ 60000 / 60000 ] 100% - 0:00:00 left 
 quantization error: 0.12811223204403668
Training completed (60000 iterations)


SOM WEIGHTS MIN/MAX 

CHECK SO MAX/MIN SCALER WORKS 

In [18]:
som_clas_200.quantization_error(df_som)
som_clas_200.topographic_error(df_som)

AttributeError: 'GeneralCLASSOM' object has no attribute 'quantization_error'

In [ ]:
som_clas_200.inspect_bmu_distribution(df_som)

SOM NEURON ANALYSIS

1. Feature-wise variation
Each feature should show variation across the map.

2. Weight planes (component maps)
Each feature is visualized across the SOM grid.

3. Neuron variance
Measures how much each neuron varies across features.

4. Dead neurons
Neurons with very low variance

5. Weight distribution
Histogram of all weights

In [ ]:
# === Hämta weights ===
weights = som_clas_200.som.get_weights()  # shape: (x, y, features)

print("=== BASIC INFO ===")
print("Shape:", weights.shape)
print("Global min:", weights.min())
print("Global max:", weights.max())
print("Global variance:", np.var(weights))

# =========================================
# 1. FEATURE-WISE ANALYS 
# =========================================
print("\n=== FEATURE-WISE RANGE ===")

n_features = weights.shape[2]

for i in range(n_features):
    w = weights[:, :, i]
    print(f"Feature {i}: min={w.min():.3f}, max={w.max():.3f}, var={np.var(w):.4f}")

# =========================================
# 2. VISUALISERA WEIGHT PLANES
# =========================================
print("\n=== VISUALIZING WEIGHT PLANES ===")

fig, axes = plt.subplots(1, n_features, figsize=(4*n_features, 4))

for i in range(n_features):
    ax = axes[i] if n_features > 1 else axes
    im = ax.imshow(weights[:, :, i], cmap="viridis")
    ax.set_title(f"Feature {i}")
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

# =========================================
# 3. NEURON VARIATION 
# =========================================
print("\n=== NEURON VARIATION ===")

# variation per neuron (över features)
neuron_variance = np.var(weights, axis=2)

print("Neuron variance stats:")
print("min:", neuron_variance.min())
print("max:", neuron_variance.max())
print("mean:", neuron_variance.mean())

plt.figure(figsize=(6,6))
plt.imshow(neuron_variance, cmap="inferno")
plt.colorbar()
plt.title("Neuron Variance (per neuron)")
plt.show()

# =========================================
# 4. DEAD / LOW-ACTIVITY NEURONS
# =========================================
print("\n=== LOW VARIANCE NEURONS ===")

threshold = np.percentile(neuron_variance, 5)

dead_mask = neuron_variance < threshold
num_dead = np.sum(dead_mask)

print(f"Low-variance neurons (<5th percentile): {num_dead}")

plt.figure(figsize=(6,6))
plt.imshow(dead_mask, cmap="gray")
plt.title("Low-variance neurons (potentially unused)")
plt.show()

# =========================================
# 5. WEIGHT DISTRIBUTION
# =========================================
print("\n=== WEIGHT DISTRIBUTION ===")

plt.figure(figsize=(6,4))
plt.hist(weights.flatten(), bins=50)
plt.title("Distribution of SOM Weights")
plt.xlabel("Weight value")
plt.ylabel("Frequency")
plt.show()

In [ ]:
cluster_map = som_clas_200.create_clusters(
    data=df_som,
    n_clusters=3,
    method="density"
)

**SOM Analysis Panel**

1. U-Matrix (Unified Distance Matrix)
The U-Matrix visualizes the average distance between neighboring neurons.

- Light areas → **similar neurons (clusters)**
- Dark areas → **large distances (cluster boundaries)**

2. Hit Map **(Activation Map)**
The hit map shows how many data samples are mapped to each neuron.

3. Cluster Map
Displays clustering results applied to SOM neurons (here using density-aware KMeans).

4. U-Matrix + Cluster Overlay
Combines structural information (U-Matrix) with clustering.

5. Feature Maps (Component Planes)

Each feature is visualized across the SOM grid using neuron weights.

Each feature map shows how that variable varies across the SOM:

- Smooth gradients → structured relationship  
- Sharp transitions → strong separation between regions  
- Uniform areas → low influence of that feature  

Feature maps help identify:

- Which variables drive clustering  
- Relationships between physiological signals  
- Correlations (e.g., HR vs HRV)  
- Regions of high/low activation  

In [ ]:
# ===== COMPLETE SOM VISUALIZATION PANEL =====

import matplotlib.pyplot as plt
import numpy as np

# ---- Prepare data ----
scaled_data = som_clas_200.scaler.transform(df_som)

u_matrix = som_clas_200.get_u_matrix()

hits = som_clas_200.som.activation_response(scaled_data)

cluster_map = som_clas_200.create_clusters(
    data=df_som,
    n_clusters=3,
    method="density"
)

weights = som_clas_200.som.get_weights()

feature_names = [
    "HR",
    "HRV_RMSSD", 
    "SC_PH", 
    "SC_RR",
    "EDA_slope"
  
]

# ---- Figure layout ----
fig = plt.figure(figsize=(16,12))

grid = fig.add_gridspec(3,4)

# ---- U-Matrix ----
ax1 = fig.add_subplot(grid[0,0])
im = ax1.imshow(u_matrix, cmap="bone_r")
ax1.set_title("U-Matrix")
ax1.axis("off")
plt.colorbar(im, ax=ax1, fraction=0.046)

# ---- Hit Map ----
ax2 = fig.add_subplot(grid[0,1])
im = ax2.imshow(hits, cmap="viridis")
ax2.set_title("Hit Map")
ax2.axis("off")
plt.colorbar(im, ax=ax2, fraction=0.046)

# ---- Cluster Map ----
ax3 = fig.add_subplot(grid[0,2])
im = ax3.imshow(cluster_map, cmap="tab10")
ax3.set_title("Cluster Map")
ax3.axis("off")
plt.colorbar(im, ax=ax3, fraction=0.046)

# ---- U-Matrix + Cluster Overlay ----
ax4 = fig.add_subplot(grid[0,3])
ax4.imshow(u_matrix, cmap="bone_r")
ax4.imshow(cluster_map, cmap="tab10", alpha=0.4)
ax4.set_title("U-Matrix + Clusters")
ax4.axis("off")

# ---- Feature Maps ----
for i in range(5):

    row = 1 + i // 3
    col = i % 3

    ax = fig.add_subplot(grid[row, col])

    im = ax.imshow(weights[:,:,i], cmap="coolwarm")
    ax.set_title(feature_names[i])
    ax.axis("off")

    plt.colorbar(im, ax=ax, fraction=0.046)

# ---- Layout ----
plt.suptitle("SOM Analysis Panel", fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
u_matrix = som_clas_200.get_u_matrix()

plt.figure(figsize=(15,15))
plt.imshow(u_matrix, cmap="bone_r")
plt.colorbar()
plt.title("U-Matrix (Neuron distances)")
plt.show()

In [ ]:
plt.figure(figsize=(10,10))

plt.imshow(u_matrix, cmap="bone_r")

plt.imshow(
    cluster_map,
    cmap="tab10",
    alpha=0.45
)

plt.colorbar(label="U-Matrix distance")

plt.title("SOM U-Matrix with Cluster Overlay")

plt.xticks([])
plt.yticks([])

plt.show()

In [ ]:
scaled_data = som_clas_200.scaler.transform(df_som)

plt.figure(figsize=(6,6))
plt.scatter(scaled_data[:, 0], scaled_data[:, 1], s=10)
plt.xlabel("HR (scaled)")
plt.ylabel("HRV_RMSSD (scaled)")
plt.title("Scatter plot of HR vs RMSSD")
plt.grid()
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import numpy as np

weights = som_clas_200.som.get_weights()
X = weights.reshape(-1, weights.shape[2])

# normalisera
X = StandardScaler().fit_transform(X)

k_values = range(2, 8)
scores = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    
    score = silhouette_score(X, labels)
    scores.append(score)
    
    print(f"k={k}, silhouette={score:.3f}")

In [ ]:
# === PREP DATA ===

import numpy as np

# 1. Hämta SOM weights
weights = som_clas_200.som.get_weights()  # (x, y, features)

# 2. Flatten → dataset
X = weights.reshape(-1, weights.shape[2])

# 3. Flatten cluster map → labels
labels = cluster_map.flatten()

# 4. Hantera ev. noise (-1 från density clustering)
if -1 in labels:
    mask = labels != -1
    X_f = X[mask]
    labels_f = labels[mask]
    print("Noise points removed:", np.sum(labels == -1))
else:
    X_f = X
    labels_f = labels

# 5. Basic checks
print("=== DATA INFO ===")
print("X shape:", X.shape)
print("Filtered X shape:", X_f.shape)
print("Unique clusters:", np.unique(labels_f))
print("Number of clusters:", len(np.unique(labels_f)))

In [ ]:
# === CLUSTER METRICS ===

from sklearn.metrics import davies_bouldin_score, silhouette_score

print("=== CLUSTER EVALUATION ===")

# Davies–Bouldin Index
dbi = davies_bouldin_score(X_f, labels_f)

# Silhouette Score
sil = silhouette_score(X_f, labels_f)

print("Davies-Bouldin Index:", round(dbi, 4))
print("Silhouette Score:", round(sil, 4))

# Tolkning (snabb guide)
if sil > 0.5:
    print("Silhouette: Strong clustering")
elif sil > 0.2:
    print("Silhouette: Moderate clustering")
else:
    print("Silhouette: Weak clustering")

In [ ]:
# === ELBOW METHOD ===

from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

wcss = []
k_range = range(1, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)  # OBS: använd X, inte X_f
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(6,4))
plt.plot(k_range, wcss, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('WCSS')
plt.title('Elbow Method (SOM Weights)')
plt.grid()
plt.show()

In [ ]:
# === CLUSTER INTERPRETATION ===

label_names = {
    0: "Neutral/Base",
    1: "Stress",
    2: "Rest/Recovery"
}

print("=== CLUSTER DISTRIBUTION ===")

unique, counts = np.unique(labels_f, return_counts=True)

for u, c in zip(unique, counts):
    name = label_names.get(u, f"Cluster {u}")
    print(f"{name}: {c} neurons")

In [ ]:
# === FEATURE ANALYSIS PER CLUSTER ===

print("=== CLUSTER FEATURE MEANS ===")

for cluster_id in np.unique(labels_f):
    mask = labels_f == cluster_id
    cluster_data = X_f[mask]
    
    name = label_names.get(cluster_id, f"Cluster {cluster_id}")
    
    print(f"\n{name}")
    print("Mean:", np.round(cluster_data.mean(axis=0), 3))
    print("Std:", np.round(cluster_data.std(axis=0), 3))

In [ ]:
# === SILHOUETTE VISUALIZATION ===

from sklearn.metrics import silhouette_samples

sample_silhouette_values = silhouette_samples(X_f, labels_f)

plt.figure(figsize=(6,4))
plt.hist(sample_silhouette_values, bins=30)
plt.title("Silhouette Value Distribution")
plt.xlabel("Silhouette value")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Lägg till cluster
df["cluster"] = som_clas_200.predict_cluster(df_som)

# Säkerställ att cluster är int
df["cluster"] = df["cluster"].astype(int)

# Definiera vilka features du vill analysera
features = ["HR", "HRV_RMSSD", "SC_PH", "SC_RR", "EDA_slope"]

# =========================
# CLUSTER PROFILES (MEAN)
# =========================
cluster_profiles = df.groupby("cluster")[features].mean()

print("Means:")
print(cluster_profiles)

# =========================
# COUNTS
# =========================
print("\nCounts:")
print(df["cluster"].value_counts())

# =========================
# STD
# =========================
print("\nStd:")
print(df.groupby("cluster")[features].std())



In [ ]:
# ================================
# FULL CLUSTER ANALYSIS CELL
# ================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from itertools import combinations

# -------------------------
# 1. DATAFRAME
# -------------------------
feature_names = ["HR", "HRV_RMSSD", "SC_PH", "SC_RR", "EDA_slope"]

df = pd.DataFrame(df_som, columns=feature_names)
df["cluster"] = som_clas_200.predict_cluster(df_som)

# -------------------------
# 2. BASIC STATS
# -------------------------
print("\n=== MEAN PER CLUSTER ===")
display(df.groupby("cluster").mean())

print("\n=== STD PER CLUSTER ===")
display(df.groupby("cluster").std())

# -------------------------
# 3. FEATURE IMPORTANCE
# -------------------------
print("\n=== FEATURE IMPORTANCE (normalized difference) ===")

global_mean = df[feature_names].mean()
global_std = df[feature_names].std()

importance = []

for c in sorted(df["cluster"].unique()):
    diff = abs(df[df.cluster == c][feature_names].mean() - global_mean) / global_std
    importance.append(diff)

importance_df = pd.DataFrame(importance, index=sorted(df["cluster"].unique()))
display(importance_df)

# -------------------------
# 4. BOXPLOTS
# -------------------------
plt.figure(figsize=(15, 10))
for i, col in enumerate(feature_names):
    plt.subplot(2, 3, i+1)
    sns.boxplot(x="cluster", y=col, data=df)
    plt.title(col)
plt.tight_layout()
plt.show()

# -------------------------
# 5. COHEN'S D (EFFECT SIZE)
# -------------------------
def cohens_d(x, y):
    return (np.mean(x) - np.mean(y)) / np.sqrt((np.std(x)**2 + np.std(y)**2) / 2)

print("\n=== COHEN'S D BETWEEN CLUSTERS ===")

for (c1, c2) in combinations(sorted(df.cluster.unique()), 2):
    print(f"\nCluster {c1} vs {c2}")
    for col in feature_names:
        d = cohens_d(df[df.cluster==c1][col], df[df.cluster==c2][col])
        print(f"{col}: {d:.2f}")

# -------------------------
# 6. PCA VISUALIZATION
# -------------------------
pca = PCA(n_components=2)
X_pca = pca.fit_transform(df[feature_names])

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=df["cluster"], palette="Set2", s=10)
plt.title("PCA of Clusters")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()



In [ ]:
# ================================
# FEATURE RANKING PER CLUSTER
# ================================

import pandas as pd
import numpy as np

feature_names = ["HR", "HRV_RMSSD", "SC_PH", "SC_RR", "EDA_slope"]

# Global statistik
global_mean = df[feature_names].mean()
global_std = df[feature_names].std()

cluster_profiles = {}
ranking_rows = []

for c in sorted(df["cluster"].unique()):
    cluster_mean = df[df.cluster == c][feature_names].mean()
    
    # Normaliserad skillnad (importance score)
    importance = (cluster_mean - global_mean) / global_std
    
    # Absolut värde för ranking
    importance_abs = importance.abs().sort_values(ascending=False)
    
    # Spara topp-features
    top_features = importance_abs.index.tolist()
    
    for i, feat in enumerate(top_features):
        ranking_rows.append({
            "Cluster": c,
            "Rank": i+1,
            "Feature": feat,
            "Importance": importance[feat],
            "Abs Importance": importance_abs[feat]
        })

ranking_df = pd.DataFrame(ranking_rows)

# Visa topp 3 per kluster
print("\n=== TOP FEATURES PER CLUSTER ===")
display(ranking_df.groupby("Cluster").head(3))

# -------------------------
# BONUS: Pivot-tabell (snygg för rapport)
# -------------------------
pivot_table = ranking_df.pivot_table(
    index="Cluster",
    columns="Rank",
    values="Feature",
    aggfunc="first"
)

pivot_table.columns = [f"Rank {c}" for c in pivot_table.columns]

print("\n=== FEATURE RANKING TABLE ===")
display(pivot_table)

In [ ]:
som_clas_200.set_cluster_labels({
    0: "Rest",
    1: "Neutral/active",
    2: "Stress"
})


In [ ]:
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

som_clas_200.export_to_c(filepath=MODEL_DIR / "som_model_200hz_ver10.h")



In [ ]:
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

model_path = MODEL_DIR / "som_model_200hz_ver11.pkl"

som_clas_200.save_model(model_path)

print("Model saved to:", model_path)

In [ ]:
X_train = df_som[features]

X_train_scaled = pd.DataFrame(
    som_clas_200.scaler.transform(X_train),
    columns=features,
    index=X_train.index
)

df_som["cluster"] = som_clas_200.predict_cluster(X_train)

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    X_train_scaled["HR"],
    X_train_scaled["HRV_RMSSD"],
    c=df_som["cluster"],
    s=5,
    alpha=0.5
)

plt.title("Train data with cluster labels (scaled)")
plt.colorbar()
plt.show()

In [ ]:
# === FEATURE DOMINANCE CHECK ===

import numpy as np
import pandas as pd

feature_names = ["HR", "HRV_RMSSD", "SC_PH", "SC_RR", "EDA_slope"]

# Varians per feature
variances = np.var(X, axis=0)

print("=== FEATURE VARIANCE ===")
for name, var in zip(feature_names, variances):
    print(f"{name}: {var:.4f}")

# Som procent (viktigare!)
total_var = np.sum(variances)
print("\n=== RELATIVE CONTRIBUTION ===")
for name, var in zip(feature_names, variances):
    print(f"{name}: {100 * var / total_var:.2f}%")

# DataFrame (snyggare output)
df_var = pd.DataFrame({
    "Feature": feature_names,
    "Variance": variances,
    "Contribution (%)": 100 * variances / total_var
})

df_var